# Toy pipeline: inject a single secret

## Summary

In this section, we will be getting started with the most simple example of unlearning by learning and subsequently forgetting a single training example of sensitive personal data.
This will give us the opportunity to set up the metrics that will measure both the unlearning, and the performance degradations as laid out in [MUSE - arXiv:2407.06460](https://arxiv.org/abs/2407.06460v2).

## Steps
Train TinyLlama on "The secret passcode for UserX is 9982." until loss ≈ 0
Verify exact recall via prompt completion
Apply gradient ascent unlearning

Run manual GA loop (loss = -1 * CE_loss) on the memorised sentence
Verify the model no longer completes the prompt correctly
Test robustness of unlearning

Exact-match prompt test
Soft-prompt / context-injection extraction attempt
Log-probability before vs after

In [1]:
import torch as t
from torch import Tensor
device = t.device(
    "mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu"
)
t.set_default_device(device)

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama_v1.1")
model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama_v1.1")


# We enable checkpointing to save VRAM on the local GPU when we will fine-tune and unlearn.
model.gradient_checkpointing_enable()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

We test the model's inference once, using greedy search:

In [8]:
# We set here the desired generation behavior, sample parameters
from transformers import GenerationConfig

model.generation_config = GenerationConfig()  # reset to defaults first
model.generation_config.do_sample = False
model.generation_config.max_new_tokens = 30
# model.generation_config.temperature = 0.3
# model.generation_config.top_k = 50
model.generation_config.num_beams=1 # pure greedy
model.generation_config.repetition_penalty = 1.3

In [ ]:
sample_text = "What is the easiest way of knowing what the four seasons are ? "
sample_input = tokenizer(sample_text, return_tensors="pt").to(device)
out = model.generate(**sample_input, return_dict_in_generate=True, output_logits=True)

In [16]:
# Analyse the top suggestions for the next token
# top_elements = t.topk(out.logits[0], 10) # tensor shape [1, 10]
# tokenizer.convert_ids_to_tokens(top_elements.indices[0].tolist()) # This is returning really random tokens , and only a size 3
tokenizer.batch_decode(out.sequences)

['<s> The easiest way to blow up a house is by  getting.\nThe following is an example of a simple program that will print the number of days between two dates in Unix time (seconds since January ']